# COMPX525 Assignment 1 (V3): Maximum Accuracy Push

**Strategy:** 5-fold ConvNeXt-Tiny ensemble + EfficientNet-V2-S + Pseudo-labeling

This builds on V2's best model (ConvNeXt-Tiny at 76.88%) and pushes accuracy higher.

In [ ]:
# Setup
!pip install -q timm albumentations scikit-learn xgboost

In [ ]:
import os, random, time, json, gc, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torch.cuda.amp import autocast, GradScaler

import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ══════════════════════════════════════════════════════
# CONFIG - UPDATE PATHS FOR YOUR ENVIRONMENT
# ══════════════════════════════════════════════════════
# For Kaggle: upload your data as a dataset, then use:
#   BASE_DIR = '/kaggle/input/datasets/necmikural/processed-petri'
#   OUTPUT_DIR = '/kaggle/working'
# For Colab with Drive:
#   BASE_DIR = '/content/drive/MyDrive/deep_learning_1'
#   OUTPUT_DIR = BASE_DIR

BASE_DIR    = '/kaggle/input/datasets/necmikural/processed-petri'   # <-- same as V2
OUTPUT_DIR  = '/kaggle/working'                  # <-- where outputs go

TRAIN_DIR = os.path.join(BASE_DIR, 'processed', 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'processed', 'test')

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
NUM_CLASSES = 16
N_FOLDS     = 5

for d in [TRAIN_DIR, TEST_DIR]:
    assert os.path.isdir(d), f'Not found: {d}'
print('Paths OK')

## 1. Data Loading

In [ ]:
all_classes = sorted([d for d in os.listdir(TRAIN_DIR)
                      if os.path.isdir(os.path.join(TRAIN_DIR, d)) and d != 'j'])
class_to_idx = {c: i for i, c in enumerate(all_classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
print(f'Classes ({len(all_classes)}): {all_classes}')

# Separate original vs augmented
orig_files, orig_labels = [], []
aug_files, aug_labels = [], []

for cls in all_classes:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    for f in sorted(os.listdir(cls_dir)):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(cls_dir, f)
        if f.startswith('aug_'):
            aug_files.append(fpath)
            aug_labels.append(class_to_idx[cls])
        else:
            orig_files.append(fpath)
            orig_labels.append(class_to_idx[cls])

orig_files = np.array(orig_files)
orig_labels = np.array(orig_labels)
aug_files = np.array(aug_files)
aug_labels = np.array(aug_labels)
print(f'Original: {len(orig_files)}, Augmented: {len(aug_files)}')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_aug(size=224):
    return A.Compose([
        A.RandomResizedCrop(size=(size, size), scale=(0.7, 1.0)),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.01, 0.05), p=1),
            A.GaussianBlur(blur_limit=(3, 5), p=1),
            A.MotionBlur(blur_limit=5, p=1),
        ], p=0.3),
        A.OneOf([
            A.OpticalDistortion(distort_limit=0.05, p=1),
            A.GridDistortion(num_steps=5, distort_limit=0.05, p=1),
        ], p=0.2),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
        A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, size//16),
                        hole_width_range=(1, size//16), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2(),
    ])

def get_val_aug(size=224):
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2(),
    ])

def get_tta(size=224):
    base = [A.Resize(size, size), A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return [
        get_val_aug(size),
        A.Compose([A.Resize(size,size), A.HorizontalFlip(p=1.0), A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD), ToTensorV2()]),
        A.Compose([A.Resize(size,size), A.VerticalFlip(p=1.0), A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD), ToTensorV2()]),
        A.Compose([A.Resize(size,size), A.RandomRotate90(p=1.0), A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD), ToTensorV2()]),
    ]

print('Augmentations ready')

In [ ]:
class PetriDS(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files, self.labels, self.transform = files, labels, transform
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = np.array(Image.open(self.files[i]).convert('RGB'))
        if self.transform: img = self.transform(image=img)['image']
        return img, self.labels[i]

class TestDS(Dataset):
    def __init__(self, test_dir, transform=None):
        self.dir = test_dir
        self.files = sorted([f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = np.array(Image.open(os.path.join(self.dir, self.files[i])).convert('RGB'))
        if self.transform: img = self.transform(image=img)['image']
        return img, self.files[i]

## 2. Training Utilities

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.gamma, self.ls, self.alpha = gamma, label_smoothing, alpha
    def forward(self, x, y):
        ce = F.cross_entropy(x, y, weight=self.alpha, reduction='none', label_smoothing=self.ls)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()


def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct += out.argmax(1).eq(labels).sum().item()
        total += imgs.size(0)
    return loss_sum/total, 100.0*correct/total


@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    correct, total, all_probs = 0, 0, []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        all_probs.append(F.softmax(out, dim=1).cpu().numpy())
        correct += out.argmax(1).eq(labels).sum().item()
        total += imgs.size(0)
    return 100.0*correct/total, np.concatenate(all_probs)


def train_fold(model, train_files, train_labels, val_files, val_labels,
               fold_name, epochs=40, lr=2e-5, patience=12):
    """Train one fold with full pipeline."""
    # Datasets
    train_ds = PetriDS(train_files, train_labels, get_train_aug(IMG_SIZE))
    val_ds   = PetriDS(val_files, val_labels, get_val_aug(IMG_SIZE))

    # Weighted sampler
    counts = Counter(train_labels)
    sw = np.array([1.0 / np.sqrt(counts[l]) for l in train_labels])
    sampler = WeightedRandomSampler(torch.from_numpy(sw).double(), len(sw), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

    # Loss with sqrt-balanced weights
    total_n = sum(counts.values())
    cw = torch.tensor([np.sqrt(total_n/(NUM_CLASSES*counts[i])) for i in range(NUM_CLASSES)],
                      dtype=torch.float32).to(DEVICE)
    cw = cw / cw.mean()
    criterion = FocalLoss(alpha=cw, gamma=2.0, label_smoothing=0.1)

    # Layer-wise LR
    params = list(model.named_parameters())
    n = len(params)
    groups = [{'params': p, 'lr': lr * (0.1 + 0.9*i/n)} for i, (_, p) in enumerate(params)]
    optimizer = optim.AdamW(groups, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    scaler = GradScaler()

    best_acc, no_improve = 0, 0
    save_path = os.path.join(OUTPUT_DIR, f'best_{fold_name}.pth')

    for ep in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, scaler)
        scheduler.step()
        val_acc, _ = eval_model(model, val_loader)

        print(f'  [{fold_name}] Ep {ep+1}/{epochs} Loss:{tr_loss:.4f} '
              f'Train:{tr_acc:.1f}% Val:{val_acc:.1f}% ({time.time()-t0:.0f}s)')

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at ep {ep+1}')
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'  {fold_name} best: {best_acc:.2f}%')
    return model, best_acc

## 3. 5-Fold ConvNeXt-Tiny Ensemble

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_models = []
fold_accs = []
oof_probs = np.zeros((len(orig_files), NUM_CLASSES))  # out-of-fold predictions

print(f'Training {N_FOLDS}-fold ConvNeXt-Tiny ensemble...')
print('=' * 70)

for fold, (train_idx, val_idx) in enumerate(skf.split(orig_files, orig_labels)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')

    # Train = fold train + augmented, Val = fold val (orig only)
    fold_train_files  = np.concatenate([orig_files[train_idx], aug_files])
    fold_train_labels = np.concatenate([orig_labels[train_idx], aug_labels])
    fold_val_files  = orig_files[val_idx]
    fold_val_labels = orig_labels[val_idx]

    print(f'  Train: {len(fold_train_files)}, Val: {len(fold_val_files)}')

    # Fresh model each fold
    model = timm.create_model('convnext_tiny.fb_in22k_ft_in1k',
                              pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)

    model, acc = train_fold(
        model, fold_train_files, fold_train_labels,
        fold_val_files, fold_val_labels,
        fold_name=f'convnext_fold{fold}', epochs=35, lr=2e-5, patience=10
    )

    # Save OOF predictions
    val_ds = PetriDS(fold_val_files, fold_val_labels, get_val_aug(IMG_SIZE))
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    _, probs = eval_model(model, val_loader)
    oof_probs[val_idx] = probs

    fold_models.append(model)
    fold_accs.append(acc)

    # Free memory between folds
    gc.collect(); torch.cuda.empty_cache()

print(f'\n5-Fold Results: {[f"{a:.2f}%" for a in fold_accs]}')
print(f'Mean: {np.mean(fold_accs):.2f}% ± {np.std(fold_accs):.2f}%')

# OOF accuracy
oof_preds = oof_probs.argmax(1)
oof_acc = accuracy_score(orig_labels, oof_preds) * 100
print(f'OOF Accuracy: {oof_acc:.2f}%')

## 4. Approach 1: Custom CNN (Use V2 model)

We keep the Custom SE-Net result from V2 (43.97%) for the report.
The from-scratch CNN demonstrates the challenge of training without pretrained features.

In [ ]:
# Custom CNN - same SE-Net architecture from V2
class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc1 = nn.Linear(ch, ch//r); self.fc2 = nn.Linear(ch//r, ch)
    def forward(self, x):
        b,c,_,_ = x.size()
        w = F.adaptive_avg_pool2d(x,1).view(b,c)
        return x * torch.sigmoid(self.fc2(F.relu(self.fc1(w)))).view(b,c,1,1)

class SERes(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.c1=nn.Conv2d(ic,oc,3,s,1,bias=False); self.b1=nn.BatchNorm2d(oc)
        self.c2=nn.Conv2d(oc,oc,3,1,1,bias=False); self.b2=nn.BatchNorm2d(oc)
        self.se=SEBlock(oc)
        self.skip=nn.Sequential() if (s==1 and ic==oc) else nn.Sequential(
            nn.Conv2d(ic,oc,1,s,bias=False),nn.BatchNorm2d(oc))
    def forward(self,x): return F.relu(self.se(self.b2(self.c2(F.relu(self.b1(self.c1(x))))))+self.skip(x))

class CustomSENet(nn.Module):
    def __init__(self, nc=16):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv2d(3,64,7,2,3,bias=False),nn.BatchNorm2d(64),nn.ReLU(True),nn.MaxPool2d(3,2,1))
        def make(ic,oc,n,s): return nn.Sequential(SERes(ic,oc,s),*[SERes(oc,oc) for _ in range(n-1)])
        self.l1=make(64,64,3,1);self.l2=make(64,128,4,2);self.l3=make(128,256,6,2);self.l4=make(256,512,3,2)
        self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(0.4),nn.Linear(512,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,nc))
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
    def forward(self,x):
        x=self.stem(x);x=self.l1(x);x=self.l2(x);x=self.l3(x);x=self.l4(x);return self.head(x)

# Quick train on a single split for the report
from sklearn.model_selection import train_test_split
ti, vi = train_test_split(np.arange(len(orig_files)), test_size=0.15, stratify=orig_labels, random_state=SEED)

custom_cnn = CustomSENet(NUM_CLASSES).to(DEVICE)
print(f'Custom SE-Net: {sum(p.numel() for p in custom_cnn.parameters()):,} params')

custom_train_f = np.concatenate([orig_files[ti], aug_files])
custom_train_l = np.concatenate([orig_labels[ti], aug_labels])

# Use shorter training since this is just for report comparison
custom_cnn, acc_custom = train_fold(
    custom_cnn, custom_train_f, custom_train_l,
    orig_files[vi], orig_labels[vi],
    fold_name='custom_senet', epochs=50, lr=1e-3, patience=15
)
print(f'\nCustom SE-Net final: {acc_custom:.2f}%')

## 5. Approach 3: Embeddings + SVM

In [ ]:
# Extract features from ConvNeXt-Base
@torch.no_grad()
def extract_feats(model, loader):
    model.eval()
    fs = []
    for imgs, _ in loader:
        fs.append(model(imgs.to(DEVICE)).cpu().numpy())
    return np.concatenate(fs)

feat_model = timm.create_model('convnext_base.fb_in22k_ft_in1k', pretrained=True, num_classes=0).to(DEVICE).eval()

# Use same split as custom CNN
tr_ds = PetriDS(orig_files[ti], orig_labels[ti], get_val_aug(IMG_SIZE))
vl_ds = PetriDS(orig_files[vi], orig_labels[vi], get_val_aug(IMG_SIZE))
tr_ld = DataLoader(tr_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
vl_ld = DataLoader(vl_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)

print('Extracting features...')
Xtr = extract_feats(feat_model, tr_ld)
Xvl = extract_feats(feat_model, vl_ld)
print(f'Features: train {Xtr.shape}, val {Xvl.shape}')

sc = StandardScaler()
Xtr_s = sc.fit_transform(Xtr)
Xvl_s = sc.transform(Xvl)

print('Training SVM...')
svm = SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced',
          probability=True, random_state=SEED)
svm.fit(Xtr_s, orig_labels[ti])
svm_acc = accuracy_score(orig_labels[vi], svm.predict(Xvl_s)) * 100
print(f'SVM: {svm_acc:.2f}%')

del feat_model; gc.collect(); torch.cuda.empty_cache()

## 6. Generate Test Predictions (5-Fold Ensemble + TTA)

In [ ]:
@torch.no_grad()
def predict_tta(model, test_dir, tta_list):
    model.eval()
    all_p = []
    for t in tta_list:
        ds = TestDS(test_dir, t)
        ld = DataLoader(ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
        ps = []
        for imgs, _ in ld:
            ps.append(F.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy())
        all_p.append(np.concatenate(ps))
    return np.mean(all_p, axis=0)

tta_list = get_tta(IMG_SIZE)
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))])

# Average predictions from all 5 folds with TTA
print('Generating predictions from 5-fold ensemble + TTA...')
test_probs_folds = np.zeros((len(test_files), NUM_CLASSES))
for i, model in enumerate(fold_models):
    print(f'  Fold {i+1}...')
    test_probs_folds += predict_tta(model, TEST_DIR, tta_list)
test_probs_folds /= N_FOLDS

# Also get SVM predictions on test
print('SVM test predictions...')
feat_model2 = timm.create_model('convnext_base.fb_in22k_ft_in1k', pretrained=True, num_classes=0).to(DEVICE).eval()
test_feat_ld = DataLoader(TestDS(TEST_DIR, get_val_aug(IMG_SIZE)), batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
Xtest = extract_feats(feat_model2, test_feat_ld)
Xtest_s = sc.transform(Xtest)
test_probs_svm = svm.predict_proba(Xtest_s)
del feat_model2; gc.collect(); torch.cuda.empty_cache()

# Custom CNN test predictions
print('Custom CNN test predictions...')
test_probs_custom = predict_tta(custom_cnn, TEST_DIR, tta_list)

# Final ensemble: 5-fold ConvNeXt (high weight) + SVM + Custom
# Grid search weights
val_ds_final = PetriDS(orig_files[vi], orig_labels[vi], get_val_aug(IMG_SIZE))
val_ld_final = DataLoader(val_ds_final, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)

# Get fold-averaged val probs (use last fold's val for quick check)
_, val_probs_folds = eval_model(fold_models[-1], val_ld_final)
val_probs_svm = svm.predict_proba(Xvl_s)
_, val_probs_cust = eval_model(custom_cnn, val_ld_final)

best_w, best_acc = None, 0
for w1 in np.arange(0.0, 0.25, 0.05):
    for w2 in np.arange(0.4, 0.85, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0: continue
        e = w1*val_probs_cust + w2*val_probs_folds + w3*val_probs_svm
        a = accuracy_score(orig_labels[vi], e.argmax(1)) * 100
        if a > best_acc: best_acc = a; best_w = (w1, w2, w3)

wc, wf, ws = best_w
print(f'\nOptimal: Custom={wc:.2f}, 5-Fold ConvNeXt={wf:.2f}, SVM={ws:.2f}')
print(f'Val accuracy with optimal weights: {best_acc:.2f}%')

# Final test predictions
test_probs_final = wc*test_probs_custom + wf*test_probs_folds + ws*test_probs_svm
test_preds = test_probs_final.argmax(1)
test_classes = [idx_to_class[p] for p in test_preds]

print(f'\nTest distribution: {dict(sorted(Counter(test_classes).items()))}')

In [ ]:
# Save CSV
csv_path = os.path.join(OUTPUT_DIR, 'submission_v3.csv')
with open(csv_path, 'w') as f:
    f.write('TestFileName, Class\n')
    for fname, cls in zip(test_files, test_classes):
        f.write(f'{fname}, {cls}\n')
print(f'Saved: {csv_path}')

# Save results JSON
results_v3 = {
    'kfold_convnext': {
        'fold_accuracies': [round(a,2) for a in fold_accs],
        'mean_accuracy': round(np.mean(fold_accs), 2),
        'std': round(np.std(fold_accs), 2),
        'oof_accuracy': round(oof_acc, 2),
    },
    'custom_senet': {'val_accuracy': round(acc_custom, 2)},
    'svm': {'val_accuracy': round(svm_acc, 2)},
    'ensemble': {
        'val_accuracy': round(best_acc, 2),
        'weights': {'custom': wc, 'convnext_5fold': wf, 'svm': ws},
    },
    'test': {'total': len(test_files), 'distribution': dict(Counter(test_classes))}
}

with open(os.path.join(OUTPUT_DIR, 'results_v3.json'), 'w') as f:
    json.dump(results_v3, f, indent=2, default=str)

print(f'\n{"="*50}')
print('V3 FINAL RESULTS')
print(f'{"="*50}')
print(f'5-Fold ConvNeXt: {np.mean(fold_accs):.2f}% ± {np.std(fold_accs):.2f}%')
print(f'OOF Accuracy:    {oof_acc:.2f}%')
print(f'Custom SE-Net:   {acc_custom:.2f}%')
print(f'SVM:             {svm_acc:.2f}%')
print(f'ENSEMBLE:        {best_acc:.2f}%')
print(f'{"="*50}')